In [ ]:
import warnings
import echopype as ep
import xarray as xr
import glob
import gc
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import sobel, binary_fill_holes, label, find_objects

In [ ]:
# Path to your folder with EK80 raw data
import_path = "./EK60_Ulysse/0710/DATA"

# Setup export folder
export_path = Path.cwd() / "EK60_Ulysse/0710/echogrammes"
export_path.mkdir(exist_ok=True, parents=True)

In [ ]:
def convert_raw_files (files_path) :
    # Get all .raw files in that directory
    raw_files = glob.glob(f"{files_path}/*.raw")

    eds = []
    for f in raw_files:
        ed = ep.open_raw(f, sonar_model="EK80")  # open each file
        eds.append(ed)

    for i, ed in enumerate(eds):
        export_file = export_path / f"ed_example_{i}.zarr" # Save to a unique Zarr file
        ed.to_zarr(export_file, overwrite=True)
    
    return eds

In [ ]:
def simrad_colormap():
    colors = [
        (0, 0, 0.3), (0, 0, 0.5), (0, 0, 0.7),
        (0, 0.4, 1.0), (0.1, 0.7, 1.0), (0.4, 1.0, 0.4),
        (1.0, 1.0, 0.0), (1.0, 0.5, 0.0), (1.0, 0.0, 0.0),
        (0.5, 0.0, 0.0)
    ]
    return mcolors.LinearSegmentedColormap.from_list("simrad", colors, N=256)


def load_and_correct_data(npz_path):
    data = np.load(npz_path)
    TS_matrix = data["data"].T
    angle_athwartship = data["angle_athwartship"].T
    beamwidth_athwartship = data["beamwidth_athwartship"]
    angle_longitudinal = data["angle_longitudinal"].T
    beamwidth_alongship = data["beamwidth_alongship"]
    data.close()

    dir_corr = -3.0103 * (
        (2 * angle_longitudinal / beamwidth_alongship) ** 2
        + (2 * angle_athwartship / beamwidth_athwartship) ** 2
        - 0.18
        * (2 * angle_longitudinal / beamwidth_alongship) ** 2
        * (2 * angle_athwartship / beamwidth_athwartship) ** 2
    )
    TS_matrix -= dir_corr
    return TS_matrix, dir_corr, angle_athwartship, angle_longitudinal


def remove_bottom(TS_matrix, margin_above=2, margin_below=50, max_depth=750):
    tsmax_idx = np.argmax(TS_matrix, axis=1)
    for i, idx in enumerate(tsmax_idx):
        start = max(idx - margin_below, 0)
        end = max(idx - margin_above, 0)
        TS_matrix[i, start:end] = -999
        TS_matrix[i, idx + 1 :] = -999
    return TS_matrix[:, :max_depth]


def detect_objects(TS, angle_longitudinal, angle_athwartship, depth_limit=350):
    mask = (TS > -80) & (TS < -15)

    # --- Détection des contours par Sobel ---
    sobel_mag = np.hypot(
        sobel(mask.astype(float), axis=0),
        sobel(mask.astype(float), axis=1),
    )
    contour_mask = sobel_mag > 0

    labeled, n_objects = label(binary_fill_holes(contour_mask), structure=np.ones((3, 3), int))
    objects_slices = find_objects(labeled)

    objects_info = []
    for j, slc in enumerate(objects_slices):
        if slc is None:
            continue
        if slc[1].start > depth_limit:
            continue

        obj_mask = labeled[slc] == (j + 1)
        ts_vals = TS[slc][obj_mask]
        if np.nanmax(ts_vals) < -70:
            continue

        ping_range = range(slc[0].start, slc[0].stop)
        ping_max_values = []

        # --- Calcul du max par ping avec filtrage des angles ---
        for p in ping_range:
            row_mask = obj_mask[p - slc[0].start, :]
            if not np.any(row_mask):
                continue

            ts_row = TS[p, slc[1].start:slc[1].stop][row_mask]
            ang_long = angle_longitudinal[p, slc[1].start:slc[1].stop][row_mask]
            ang_athw = angle_athwartship[p, slc[1].start:slc[1].stop][row_mask]

            # Filtrage angulaire ±6°
            valid_mask = (np.abs(ang_long) <= 5) 
            if not np.any(valid_mask):
                continue

            ts_valid = ts_row[valid_mask]
            if len(ts_valid) > 0:
                ping_max_values.append(np.nanmax(ts_valid))

        if len(ping_max_values) == 0:
            continue

        ts_med = np.nanmedian(ping_max_values)
        ts_mean = np.nanmean(ping_max_values)
        ts_max = np.nanmax(ping_max_values)

        objects_info.append({
            "slice": slc,
            "ts_med": ts_med,
            "ts_mean": ts_mean,
            "ts_max": ts_max,
        })

    return objects_info, contour_mask


def plot_TS(TS, objects_info, contour_mask, ping_times, depths, export_file, title):
    simrad_cmap = simrad_colormap()
    fig, ax = plt.subplots(figsize=(14, 6))
    norm = mcolors.PowerNorm(vmin=-80, vmax=-20, gamma=0.8)

    im = ax.imshow(
        TS.T,
        aspect="auto",
        cmap=simrad_cmap,
        norm=norm,
        extent=[ping_times[0], ping_times[-1], depths[-1], depths[0]],
        origin="upper",
    )
    ax.contour(
        contour_mask.T,
        levels=[0.5],
        colors="white",
        linewidths=0.5,
        extent=[ping_times[0], ping_times[-1], depths[-1], depths[0]],
        origin="upper",
    )

    for obj in objects_info:
        slc = obj["slice"]
        ping_center = np.mean(ping_times[slc[0].start:slc[0].stop])
        depth_center = np.mean(depths[slc[1].start:slc[1].stop])
        ax.text(
            ping_center,
            depth_center + 5,
            f"{obj['ts_med']:.1f} dB",
            color="white",
            fontsize=5,
            ha="left",
            va="top",
            bbox=dict(facecolor="black", alpha=0.4, boxstyle="round,pad=0.2"),
        )

    ax.set_xlabel("Ping index")
    ax.set_ylabel("Depth (index)")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="TS (dB)")
    fig.savefig(export_file, bbox_inches="tight", dpi=200)
    plt.close(fig)


def plot_histograms(ts_list, export_file, title):
    a, b = 19.1, -63.8
    L = 10 ** ((np.array(ts_list) - b) / a)

    plt.figure(figsize=(14, 4))
    plt.subplot(1, 2, 1)
    plt.hist(ts_list, bins=40, color="mediumseagreen", edgecolor="black")
    plt.xlabel("TS_med (dB)")
    plt.ylabel("Nombre d’objets")
    plt.title("Histogramme TS_med")
    plt.grid(True, linestyle="--", alpha=0.5)

    plt.subplot(1, 2, 2)
    plt.hist(L, bins=40, color="cornflowerblue", edgecolor="black")
    plt.xlabel("Longueur L (cm)")
    plt.ylabel("Nombre d’objets")
    plt.title("Histogramme Longueur")
    plt.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(export_file, bbox_inches="tight", dpi=200)
    plt.close()


def plot_TS_from_folder(npz_folder, export_folder, pings_per_plot=1000):
    npz_folder = Path(npz_folder)
    export_folder = Path(export_folder)
    export_folder.mkdir(parents=True, exist_ok=True)
    npz_files = sorted(npz_folder.glob("*.npz"))

    for npz_path in npz_files:
        print(f"\n🔹 Fichier : {npz_path.name}")
        TS_matrix, dir_corr, angle_longitudinal, angle_athwartship = load_and_correct_data(npz_path)
        TS_matrix = remove_bottom(TS_matrix)

        n_pings, n_depths = TS_matrix.shape
        n_slices = int(np.ceil(n_pings / pings_per_plot))
        all_ts_med = []

        for s in range(n_slices):
            start_ping = s * pings_per_plot
            end_ping = min((s + 1) * pings_per_plot, n_pings)
            TS = TS_matrix[start_ping:end_ping, :]
            ping_times = np.arange(start_ping, end_ping)
            depths = np.arange(n_depths)

            objects_info, contour_mask = detect_objects(TS,angle_longitudinal[start_ping:end_ping, :],angle_athwartship[start_ping:end_ping, :])
            print(f"  ▫️ Partie {s+1}/{n_slices} : {len(objects_info)} objets")

            for o in objects_info:
                all_ts_med.append(o["ts_max"])

            fig_file = export_folder / f"{npz_path.stem}_TS_plot_{s+1}.png"
            plot_TS(TS, objects_info, contour_mask, ping_times, depths, fig_file,
                    f"{npz_path.stem} - Partie {s+1}/{n_slices}")

            gc.collect()

        if all_ts_med:
            hist_file = export_folder / f"{npz_path.stem}_hist.png"
            plot_histograms(all_ts_med, hist_file, npz_path.stem)
            print(f"📊 Histogramme sauvegardé : {hist_file}")
        else:
            print("⚠️ Aucun objet détecté.")


In [ ]:
plot_TS_from_folder("EK60_Ulysse/0710/echo_irene/DATA","EK60_Ulysse/0710/echo_irene/echogrammes")